In [ ]:
###新方案
from sshtunnel import SSHTunnelForwarder
import pymysql
import pandas as pd

def get_data(sql):
    # 跳板机和数据库信息
    ssh_host = '1.14.130.59'
    ssh_port = 22
    ssh_user = 'jumpuser'
    ssh_password = ''

    db_host = '172.16.16.16'  # 不是localhost
    db_port = 3306
    db_user = 'duanyang'
    db_password = ''

    with SSHTunnelForwarder(
        (ssh_host, ssh_port),
        ssh_username=ssh_user,
        ssh_password=ssh_password,
        remote_bind_address=(db_host, db_port),
        local_bind_address=('127.0.0.1', 3307)  # 本地端口可自定义
    ) as tunnel:
        conn = pymysql.connect(
            host='127.0.0.1',
            port=3307,
            user=db_user,
            passwd=db_password,
            charset='utf8'
        )
        cursor = conn.cursor()
        cursor.execute(sql)
        data = cursor.fetchall()
        column_names = [i[0] for i in cursor.description]
        df = pd.DataFrame(data, columns=column_names)
        conn.close()
        return df

In [ ]:
sql = """
select t1.series as 系列,
       t2.产地,
       t3.年月,
       t4.美金采购价格,
       t5.搬砖人民币采购价格,
       ifnull(t5.搬砖人民币采购价格,t6.CPG采购价) as 采购价,
       t6.CPG采购价,
       t7.采购报价,
       er.rate as 汇率,
       t4.美金采购价格 * er.rate as 美金采购价格折算人民币
from (
    select distinct dcdss.品类 as series
    from cpda_temp1.dy_cpg_dows_series_sku dcdss
     ) t1
left join (
    select distinct case when dk.产地 in ('美国','中东','欧洲') then '非泰国' else dk.产地 end as 产地
    from cpda_temp1.dy_k3 dk
) t2 on 1=1
left join (
    select distinct date_format(ob.order_create_date,'%%Y-%%m') as 年月
    from cpda_temp1.orders_board ob
    where ob.order_create_date >= '2020-01-01' and
          ob.order_create_date <= '2024-12-20'
) t3 on 1=1
left join (
select

       t.系列,
       t.产地,
       t.年月,
       sum(t.金额)/sum(t.吨数)+0 as 美金采购价格
from (
select date_format(dk.日期,'%%Y-%%m') as 年月,
       date_format(case when dk.产地 in ('泰国') then date_add(dk.日期,interval 1 month)
              when dk.产地 in ('中东') then date_add(dk.日期,interval 2 month)
              when dk.产地 in ('欧洲') then date_add(dk.日期,interval 4 month)
              when dk.产地 in ('美国') then date_add(dk.日期,interval 6 month)
           end, '%%%%Y-%%%%m') as 交货时间,
       dk.合同号,
       dk.牌号,
       dk.产地,
       dcdss.series as 系列,
       dk.吨数,
       dk.价格,
       dk.金融 as 金额
from cpda_temp1.dy_k3 dk
left join (
    select dcdss.牌号 as designation,
           dcdss.品类 as series
    from cpda_temp1.dy_cpg_dows_series_sku dcdss
) dcdss on dk.牌号 = dcdss.designation
union all
select date_format(po.created_at,'%%Y-%%m') as 年月,
       date_format(case when c.name_cn = '泰国' then date_add(po.created_at,interval 1 month)
              when c.name_cn = '沙特阿拉伯' then date_add(po.created_at,interval 2 month)
              when c.name_cn = '欧盟' then date_add(po.created_at,interval 4 month)
              when c.name_cn in ('北美','美国') then date_add(po.created_at,interval 6 month)
           end, '%%Y-%%m') as 交货时间,
       po.purchase_contract_no as 合同号,
       p.designation as 牌号,
       case when c.name_cn in ('北美','美国') then '非泰国'
           when c.name_cn = '沙特阿拉伯' then '非泰国'
           when c.name_cn = '欧盟' then '非泰国'
              when c.name_cn = '泰国' then '泰国'
              else c.name_cn
           end as 产地,
       dcdss.series as 系列,
         poi.weight as 吨数,
         case when poi.currency = 'CNY' then poi.price/er.rate
             when poi.currency = 'USD' then poi.price
             end as 价格,
         case when poi.currency = 'CNY' then poi.amount/er.rate
             when poi.currency = 'USD' then poi.amount
             end as 金额
from orderdb.purchase_order po
left join orderdb.purchase_order_item poi on po.id = poi.purchase_order_id
left join productdb.products p on poi.product_id = p.id
left join productdb.countries c on p.origin_country = c.country_code
left join (
    select dcdss.牌号 as designation,
           dcdss.品类 as series
    from cpda_temp1.dy_cpg_dows_series_sku dcdss
) dcdss on p.designation = dcdss.designation
left join cpda_temp1.dy_exchange_rate er on date_format(po.created_at,'%%Y-%%m') = date_format(er.date,'%%%%Y-%%%%m')
where date(po.created_at) >= '2023-01-01'
and date(po.created_at) <= '2024-12-20'
and  poi.manufacturer_name = '陶氏'
and po.status in (2,4,5,6)
and poi.weight > 0
and poi.currency = 'USD'
and po.purchase_contract_no regexp 'SJSC|CPG') t
where t.系列 is not null
group by t.年月, t.系列,t.产地
order by t.系列,t.产地,t.年月) t4 on t1.series = t4.系列 and
                                 t2.产地 = t4.产地 and
                                 t3.年月 = t4.年月
left join (
select
       t.系列,
       t.产地,
       t.年月,
       sum(t.金额)/sum(t.吨数) + 0 as 搬砖人民币采购价格
from (select date_format(ob.order_create_date, '%%Y-%%m') as 年月,
             ob.order_no                                as 合同号,
             p.designation                              as 牌号,
             case
                 when c.name_cn in ('北美', '加拿大','美国') then '非泰国'
                 when c.name_cn = '沙特阿拉伯' then '非泰国'
                 when c.name_cn in ('欧盟','荷兰') then '非泰国'
                 when c.name_cn = '泰国' then '泰国'
                 else c.name_cn
                 end                                    as 产地,
             dcdss.series                               as 系列,
             ob.weight                                  as 吨数,
             ob.purchase_price                          as 价格,
             ob.purchase_price * ob.weight              as 金额
      from cpda_temp1.orders_board ob
               left join productdb.products p on ob.product_id = p.id
               left join productdb.countries c on p.origin_country = c.country_code
               left join (
                            select dcdss.牌号 as designation,
                                   dcdss.品类 as series
                            from cpda_temp1.dy_cpg_dows_series_sku dcdss
                        ) dcdss on p.designation = dcdss.designation
      where ob.STATUS in (7, 8, 9, 10, 11, 15)
        and ob.is_agent_purchase = '否'
        and date(ob.order_create_date) >= '2020-01-01'
        and date(ob.order_create_date) <= '2024-12-20'
        and ob.order_type = '非资源货'
        and ob.weight > 0) t
where t.系列 is not null
group by t.年月, t.系列, t.产地
order by t.系列, t.产地, t.年月) t5 on t1.series = t5.系列 and
                                 t2.产地 = t5.产地 and
                                 t3.年月 = t5.年月
left join (
    select t.品类,
           case when t.产地 regexp '泰国' then '泰国'
               else '非泰国'
               end as 修改后产地,
        t.年月,
        avg(t.人民币价格RMB) as CPG采购价
    from cpda_temp1.dy_temp_241223_3 t
    where t.人民币价格RMB is not null and
          t.人民币价格RMB <> ''
    group by t.品类,修改后产地,t.年月
) t6 on t1.series = t6.品类 and
        t2.产地 = t6.修改后产地 and
        t3.年月 = t6.年月
left join (
select
       t.系列,
       t.产地,
       t.年月,
       sum(t.金额)/sum(t.吨数) + 0 as 采购报价
from (select date_format(ob.order_create_date, '%%Y-%%m') as 年月,
             ob.order_no                                as 合同号,
             p.designation                              as 牌号,
             case
                 when c.name_cn in ('北美', '加拿大','美国') then '非泰国'
                 when c.name_cn = '沙特阿拉伯' then '非泰国'
                 when c.name_cn in ('欧盟','荷兰') then '非泰国'
                 when c.name_cn = '泰国' then '泰国'
                 else c.name_cn
                 end                                    as 产地,
             dcdss.series                               as 系列,
             ob.weight                                  as 吨数,
             ob.purchase_price                          as 价格,
             ob.list_price * ob.weight                                  as 金额
      from cpda_temp1.orders_board ob
               left join productdb.products p on ob.product_id = p.id
               left join productdb.countries c on p.origin_country = c.country_code
               left join (
                            select dcdss.牌号 as designation,
                                   dcdss.品类 as series
                            from cpda_temp1.dy_cpg_dows_series_sku dcdss
                        ) dcdss on p.designation = dcdss.designation
      where ob.STATUS in (7, 8, 9, 10, 11, 15)
        and ob.is_agent_purchase = '否'
        and date(ob.order_create_date) >= '2020-01-01'
        and date(ob.order_create_date) <= '2024-12-20'
        and ob.order_type <> '无采购单'
        and ob.weight > 0) t
where t.系列 is not null
group by t.年月, t.系列, t.产地
order by t.系列, t.产地, t.年月) t7 on t1.series = t7.系列 and
                                 t2.产地 = t7.产地 and
                                 t3.年月 = t7.年月
left join cpda_temp1.dy_exchange_rate er on t3.年月 = date_format(er.date,'%%Y-%%m')
order by t1.series, t2.产地, t3.年月;
"""

df_origin = get_data(sql)

df_origin


In [ ]:
#df_origin[年月]转换为datetime格式
df_origin['年月'] = pd.to_datetime(df_origin['年月'])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

year_list = [2020,2021,2022,2023,2024]

series_list = df_origin['系列'].unique()

country_list = df_origin['产地'].unique()

index_list = ['采购价']

for i in range(len(series_list)):
    for j in range(len(country_list)):

        df = df_origin[(df_origin['系列'] == series_list[i]) & (df_origin['产地'] == country_list[j])]

        for k in range(len(index_list)):
            df_sub = df[['年月',index_list[k]]]

            df_sub['年'] = df_sub['年月'].dt.year
            df_sub['月'] = df_sub['年月'].dt.month

            target_df = pd.DataFrame()

            target_df['月'] = [i for i in range(1,13)]

            for year in year_list:

                df_year = df_sub[df_sub['年']==year]

                #删除列年
                df_year = df_year.drop(columns=['年','年月'])

                #将df_year与target_df合并，以target_df为基准，日期列相同的行合并
                target_df = pd.merge(target_df,df_year,on='月',how='left')
                #加权平均价列重命名为对应年份
                target_df = target_df.rename(columns={index_list[k]:str(year)})


            df_sub_2 = target_df.copy()

            #设置显示中文字体
            plt.rcParams['font.sans-serif']=['SimHei']
            plt.rcParams['axes.unicode_minus'] = False

            # 确保日期列是字符串格式
            df_sub_2['月'] = df_sub_2['月'].astype(str)

            # 创建一个代表年份的临时日期（例如 2000 年），用于绘图
            temp_year = '2000-'
            df_sub_2['月'] = pd.to_datetime(temp_year + df_sub_2['月'])

            # 设置日期为索引
            df_sub_2 = df_sub_2.set_index('月')

            # 先填充所有年份的空值，除了2024年
            df_sub_2['2020'] = df_sub_2['2020'].fillna(method='ffill')
            df_sub_2['2021'] = df_sub_2['2021'].fillna(method='ffill')
            df_sub_2['2022'] = df_sub_2['2022'].fillna(method='ffill')
            df_sub_2['2023'] = df_sub_2['2023'].fillna(method='ffill')


            # 特别处理 2024 年的数据
            #index_2024_01_26 = pd.to_datetime(temp_year + '01-26')
            #df_sub_2['2024'] = df_sub_2['2024'].where(df_sub_2.index <= index_2024_01_26)

            # 绘制折线图
            plt.figure(figsize=(15, 8))
            for column in df_sub_2.columns:
                line = plt.plot(df_sub_2.index, df_sub_2[column], label=column)

                # 获取当前线条的颜色
                line_color = line[0].get_color()

                # 找到每年的最高点和最低点
                max_price = df_sub_2[column].max()
                min_price = df_sub_2[column].min()
                max_date = df_sub_2[column].idxmax()
                min_date = df_sub_2[column].idxmin()

                # 在最高点和最低点标注价格，不保留小数位
                plt.annotate(f'{max_price:.0f}', xy=(max_date, max_price), xytext=(max_date, max_price),
                             arrowprops=dict(facecolor=line_color, shrink=0.05), ha='center')
                plt.annotate(f'{min_price:.0f}', xy=(min_date, min_price), xytext=(min_date, min_price),
                             arrowprops=dict(facecolor=line_color, shrink=0.05), ha='center')

            # 设置图表标题和坐标轴标签
            plt.title(series_list[i]+country_list[j]+index_list[k]+'价格趋势')
            plt.xlabel('年月')
            plt.ylabel('价格')

            # 设置 x 轴的格式器和定位器
            plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
            plt.gca().xaxis.set_major_locator(mdates.MonthLocator())

            # 旋转日期标签以便阅读
            plt.gcf().autofmt_xdate()

            # 显示图例
            plt.legend()

            # 显示图表
            plt.show()